# Disease State Prediction: PCA vs Image Embeddings

Sample-level disease prediction on Kidney / Bowel / Brain, comparing PCA with PCA + image embeddings (GPFM / UNI / UNI2-h / GigaPath).


In [ ]:
import os, sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, matthews_corrcoef, roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════
# Import core functions from the synthetic source files
# ══════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════
# Dataset selection (Kidney / Bowel / Brain)
# ══════════════════════════════════════════════════════════
DATASETS = ["Kidney", "Bowel", "Brain"]  # the three datasets supported by synthetic
RESULTS = {}  # {dataset: DataFrame}

SPG_DISEASE_DIR = Path("../../synthetic/src/05_downstream/sample_disease_prediction/Kidney")
sys.path.insert(0, str(SPG_DISEASE_DIR.resolve()))
from predict import (
    sample_feature, fit_pca, get_fold_pca,
    classifier_params, sample_package_weights,
    subset_sample, dense, resolve_path, load_yaml,
)

print("All dependencies imported successfully")
"print(f\"  predict.py: {SPG_DISEASE_DIR / 'predict.py'}\")"

## Configuration

Parameters are read from `configs/sample_disease_prediction/{Dataset}/downstream.yaml`.


In [ ]:
# ══════════════════════════════════════════════════════════
# Read parameters from the synthetic config files
# ══════════════════════════════════════════════════════════
HEST_DATA_DIR = Path("../../data/HEST")
OUTPUT_DIR = Path("../outputs/sample_disease_prediction")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
N_FOLDS = 5
RANDOM_SEED = 42

POS_LABEL_MAP = {"Kidney": "SCCRCC", "Bowel": "Cancer", "Brain": "EPM"}
SCORE_COL_MAP = {k: f"score_{v}" for k, v in POS_LABEL_MAP.items()}

print(f"General configuration:")
print(f"  CV: {N_FOLDS} folds, seed={RANDOM_SEED}")
"print(f\"  Datasets: {DATASETS}\")"

## Data Loading

Load the sample packages prepared by the synthetic pipeline.


In [ ]:
def load_dataset(dataset_name: str):
    """Load a single dataset and return (real, splits). Consistent with synthetic prepare_sample_disease_*."""
    interim_path = resolve_path(f"configs/../data/02_interim/sample_disease_prediction/{dataset_name}/{dataset_name.lower()}_samples.h5ad")
    splits_path = resolve_path(f"configs/../data/02_interim/sample_disease_prediction/{dataset_name}/splits.csv")

    if interim_path.exists() and splits_path.exists():
        real = ad.read_h5ad(interim_path)
        splits = pd.read_csv(splits_path)
        splits["sample_id"] = splits["sample_id"].astype(str)
        print(f"  Loading preprocessed data: {interim_path}")
        return real, splits

    # Build from the raw HEST data
    print(f"  Preprocessed data not found; building from HEST...")
    meta = pd.read_csv(HEST_DATA_DIR / "meta_df_t_update.csv")
    meta = meta[meta["organ"] == dataset_name]

    def disease_label(row):
        update = str(row.get("disease_update", "") or "").strip()
        oncotree = str(row.get("oncotree_code", "") or "").strip()
        state = str(row.get("disease_state", "") or "").strip()
        if dataset_name == "Kidney":
            if update.upper() == "SCCRCC" or oncotree.upper() == "SCCRCC" or state.lower() == "cancer":
                return "SCCRCC"
            if update.lower() == "healthy" or state.lower() == "healthy":
                return "Healthy"
        elif dataset_name == "Bowel":
            cancer_codes = {"COAD", "READ", "COADREAD"}
            if update.upper() in cancer_codes or oncotree.upper() in cancer_codes or state.lower() == "cancer":
                return "Cancer"
            if update.lower() == "healthy" or state.lower() == "healthy":
                return "Healthy"
        elif dataset_name == "Brain":
            if update.upper() == "EPM" or oncotree.upper() == "EPM":
                return "EPM"
            if update.lower() == "healthy" or state.lower() == "healthy":
                return "Healthy"
        return "Other"

    meta["disease_label"] = meta.apply(disease_label, axis=1)
    pos_labels_list = {"Kidney": ["SCCRCC", "Healthy"], "Bowel": ["Cancer", "Healthy"], "Brain": ["EPM", "Healthy"]}
    meta = meta[meta["disease_label"].isin(pos_labels_list[dataset_name])]

    adatas = []
    for _, row in meta.iterrows():
        sid = row["id"]
        path = HEST_DATA_DIR / "raw" / "st" / f"{sid}.h5ad"
        if not path.exists():
            continue
        a = ad.read_h5ad(path)
        if "in_tissue" in a.obs.columns:
            a = a[a.obs["in_tissue"].astype(bool)].copy()
        a.obs["sample_id"] = sid
        a.obs["disease_label"] = row["disease_label"]
        a.obs["patient"] = str(row.get("patient", sid))
        adatas.append(a)

    common_genes = set(adatas[0].var_names)
    for a in adatas[1:]:
        common_genes &= set(a.var_names)
    common_genes = sorted(common_genes)
    aligned = [a[:, common_genes].copy() for a in adatas]
    real = ad.concat(aligned, merge="same")

    patients = real.obs[["sample_id", "patient", "disease_label"]].drop_duplicates()
    from sklearn.model_selection import StratifiedGroupKFold
    sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    split_rows = []
    for fold, (train_idx, test_idx) in enumerate(sgkf.split(patients["sample_id"], patients["disease_label"], groups=patients["patient"])):
        for idx in train_idx:
            split_rows.append({"sample_id": patients.iloc[idx]["sample_id"], "fold": fold, "split": "train"})
        for idx in test_idx:
            split_rows.append({"sample_id": patients.iloc[idx]["sample_id"], "fold": fold, "split": "test"})
    splits = pd.DataFrame(split_rows)
    print(f"  Build complete: {real.n_obs} spots, {real.obs['sample_id'].nunique()} samples")
    return real, splits

## Loading Image Embeddings

Load the image embedding of each sample for every foundation model.


In [ ]:
# ══════════════════════════════════════════════════════════
# Embedding config: all 4 image foundation models
# ══════════════════════════════════════════════════════════
EMBED_CONFIG = [
    ("gpfm",     "GPFM",    1024),
    ("uni",      "UNI",     1024),
    ("uni2_h",   "UNI2-h",  1536),
    ("gigapath", "GigaPath", 1536),
]


def load_single_embedding(real: ad.AnnData, emb_type: str) -> dict:
    """Load one GPFM/UNI/UNI2-h/GigaPath embedding, aligned by barcode.

    Returns {sid: (embedding_array, [full_kept_barcodes])}.
    full_kept_barcodes carries the sample prefix (e.g., "INT1:AAACAGAGCGACTCCT-1"),
    and can be used directly to filter AnnData obs_names.
    """
    import h5py
    sample_ids = real.obs["sample_id"].unique()
    emb_data = {}
    missing = 0
    for sid in sample_ids:
        p = HEST_DATA_DIR / "pkl" / f"{sid}_{emb_type}_emb.pkl"
        patch_h5 = HEST_DATA_DIR / "raw" / "patches" / f"{sid}.h5"
        if not p.exists() or not patch_h5.exists():
            missing += 1
            continue
        with h5py.File(patch_h5, "r") as f:
            patch_barcodes = [b[0].decode() if isinstance(b[0], bytes) else str(b[0]) for b in f["barcode"][:]]
        full_emb = torch.load(p, weights_only=True, map_location="cpu").numpy().astype("float32")
        assert len(patch_barcodes) == full_emb.shape[0]
        # Patch barcodes have no sample prefix; AnnData barcodes carry the "sid:" prefix
        prefix = f"{sid}:"
        sample_barcodes_raw = set(real.obs_names[real.obs["sample_id"] == sid].astype(str))
        sample_barcodes_clean = set()
        for bc in sample_barcodes_raw:
            sample_barcodes_clean.add(bc[len(prefix):] if bc.startswith(prefix) else bc)
        keep_idx = [i for i, bc in enumerate(patch_barcodes) if bc in sample_barcodes_clean]
        if keep_idx:
            kept_bcs_full = [f"{prefix}{patch_barcodes[i]}" for i in keep_idx]
            emb_data[sid] = (full_emb[keep_idx], kept_bcs_full)
        else:
            missing += 1
    if emb_data:
        dim = emb_data[list(emb_data.keys())[0]][0].shape[1]
        print(f"  Loaded {emb_type}: {len(emb_data)}/{len(sample_ids)} samples, dim={dim}d")
    return emb_data


def load_all_embeddings(real: ad.AnnData) -> dict:
    """Load all EMBED_CONFIG embeddings; returns {emb_type_key: {sid: (emb, bcs)}}."""
    all_emb = {}
    for key, name, _ in EMBED_CONFIG:
        ed = load_single_embedding(real, key)
        if ed:
            all_emb[key] = ed
            print(f"    → {name}: {len(ed)} samples succeeded")
        else:
            print(f"    → {name}: no embeddings available, skipping")
    return all_emb

## Feature Construction

PCA(30) quantile + statistics baseline, and the same statistics computed on PCA + image embeddings.


In [ ]:
def sample_feature_with_embedding(
    package: ad.AnnData,
    weights: np.ndarray,
    pca: "PCA",
    genes: pd.Index,
    quantiles: list[float],
    embedding: np.ndarray | None = None,
) -> np.ndarray:
    """Same as sample_feature, but concatenates the embedding per spot after PCA projection before computing statistics."""
    package = package[:, list(genes)].copy()
    x = dense(package.X).astype(np.float32)
    z = pca.transform(x)  # (n_spots, n_pcs)
    if embedding is not None:
        # embedding must match the number of spots in package (already aligned by barcode)
        assert z.shape[0] == embedding.shape[0],             f"PCA spots ({z.shape[0]}) != embedding spots ({embedding.shape[0]})"
        z = np.hstack([z, embedding])  # (n_spots, n_pcs + emb_dim)
    qs = np.quantile(z, quantiles, axis=0).reshape(-1)
    coords = np.asarray(package.obsm["spatial"][:, :2], dtype=np.float64)
    coord_stats = np.concatenate([
        coords.mean(axis=0), coords.std(axis=0),
        coords.max(axis=0) - coords.min(axis=0),
    ])
    weights = np.asarray(weights, dtype=np.float64)
    weights = weights / max(float(weights.sum()), 1e-12)
    mean = np.average(z, axis=0, weights=weights)
    sd = np.sqrt(np.maximum(np.average((z - mean[None, :]) ** 2, axis=0, weights=weights), 0.0))
    return np.concatenate([mean, sd, qs, coord_stats]).astype(np.float32)

print("Feature functions defined")
print("  sample_feature → synthetic baseline (reused directly)")
print("  sample_feature_with_embedding → per-spot embedding concatenation after PCA → statistics")

## Running 5-Fold Cross-Validation

For each fold and classifier, train and predict, then save `predictions.csv`, `packages.csv`, `manifest.json`.


In [ ]:
import json

for dataset_name in DATASETS:
    print(f"\n{'#'*60}\n# Dataset: {dataset_name}\n{'#'*60}")

    # Load data
    real, splits = load_dataset(dataset_name)

    # Load all 4 embeddings
    emb_data_dict = load_all_embeddings(real)

    # Read parameters from the config file
    cfg = load_yaml(f"configs/sample_disease_prediction/{dataset_name}/downstream.yaml")
    n_pcs = int(cfg.get("n_pcs", 30))
    quantiles = [float(q) for q in cfg.get("feature_quantiles", [0.1, 0.5, 0.9])]
    classifiers = cfg.get("classifiers", ["LR", "XGBoost", "MLP"])
    n_jobs = int(cfg.get("n_jobs", 4))
    pos_label = POS_LABEL_MAP[dataset_name]
    score_col = SCORE_COL_MAP[dataset_name]

    fold_values = sorted(splits["fold"].unique().tolist())

    for fold in fold_values:
        split_fold = splits[splits["fold"] == fold]
        train_samples = sorted(split_fold.loc[split_fold["split"] == "train", "sample_id"].astype(str).unique())
        test_samples = sorted(split_fold.loc[split_fold["split"] == "test", "sample_id"].astype(str).unique())
        print(f"\n  Fold {fold}: {len(train_samples)} train, {len(test_samples)} test")

        pca, genes = fit_pca(real, train_samples, n_pcs=n_pcs)

        # ── PCA baseline matrix ──
        x_tr, y_tr = [], []
        for sid in train_samples:
            rs = subset_sample(real, sid)
            w = np.full(rs.n_obs, 1.0 / max(1, rs.n_obs), dtype=np.float64)
            x_tr.append(sample_feature(rs, w, pca, genes, quantiles, cache_key=None))
            y_tr.append(str(rs.obs["disease_label"].iloc[0]))

        x_te, y_te, sid_te = [], [], []
        for sid in test_samples:
            rs = subset_sample(real, sid)
            w = np.full(rs.n_obs, 1.0 / max(1, rs.n_obs), dtype=np.float64)
            x_te.append(sample_feature(rs, w, pca, genes, quantiles, cache_key=None))
            y_te.append(str(rs.obs["disease_label"].iloc[0]))
            sid_te.append(sid)

        X_tr = np.vstack(x_tr); y_tr_a = np.asarray(y_tr)
        X_te = np.vstack(x_te); y_te_a = np.asarray(y_te)
        enc = LabelEncoder()
        y_tr_enc = enc.fit_transform(y_tr_a)
        y_te_enc = enc.transform(y_te_a)
        pi = int(np.where(enc.classes_ == pos_label)[0][0]) if pos_label in enc.classes_ else 1

        # ── Method list: PCA + each embedding method ──
        METHODS = [("PCA", None, None)] + [
            (f"PCA+{name}", key, emb_data_dict.get(key))
            for key, name, _ in EMBED_CONFIG
            if key in emb_data_dict
        ]

        for method_name, emb_key, emb_data in METHODS:
            is_pca = method_name == "PCA"
            if not is_pca and emb_data is None:
                continue

            if is_pca:
                cur_X_tr, cur_y_tr = X_tr, y_tr_a
                cur_X_te, cur_y_te = X_te, y_te_a
                cur_sid_te = sid_te
                cur_enc = enc
                cur_pi = pi
                cur_pos_label = pos_label
            else:
                # PCA + embedding
                x_tr_e, y_tr_e = [], []
                for sid in train_samples:
                    if sid not in emb_data:
                        continue
                    emb_arr, kept_bcs = emb_data[sid]
                    rs = subset_sample(real, sid)
                    rs_f = rs[rs.obs_names.astype(str).isin(kept_bcs)].copy()
                    if rs_f.n_obs < 2:
                        continue
                    w = np.full(rs_f.n_obs, 1.0 / max(1, rs_f.n_obs), dtype=np.float64)
                    x_tr_e.append(sample_feature_with_embedding(rs_f, w, pca, genes, quantiles, embedding=emb_arr))
                    y_tr_e.append(str(rs_f.obs["disease_label"].iloc[0]))
                x_te_e, y_te_e, sid_te_e = [], [], []
                for sid in test_samples:
                    if sid not in emb_data:
                        continue
                    emb_arr, kept_bcs = emb_data[sid]
                    rs = subset_sample(real, sid)
                    rs_f = rs[rs.obs_names.astype(str).isin(kept_bcs)].copy()
                    if rs_f.n_obs < 2:
                        continue
                    w = np.full(rs_f.n_obs, 1.0 / max(1, rs_f.n_obs), dtype=np.float64)
                    x_te_e.append(sample_feature_with_embedding(rs_f, w, pca, genes, quantiles, embedding=emb_arr))
                    y_te_e.append(str(rs_f.obs["disease_label"].iloc[0]))
                    sid_te_e.append(sid)
                if not x_tr_e or not x_te_e:
                    continue
                cur_X_tr = np.vstack(x_tr_e)
                cur_X_te = np.vstack(x_te_e)
                cur_y_tr = np.asarray(y_tr_e)
                cur_y_te = np.asarray(y_te_e)
                cur_sid_te = sid_te_e
                cur_enc = LabelEncoder()
                cur_y_tr_enc = cur_enc.fit_transform(cur_y_tr)
                cur_y_te_enc = cur_enc.transform(cur_y_te)
                cur_pi = int(np.where(cur_enc.classes_ == cur_pos_label)[0][0]) if cur_pos_label in cur_enc.classes_ else 1

            # ── Unified training & prediction ──
            cur_y_tr_enc = cur_enc.fit_transform(cur_y_tr) if not is_pca else y_tr_enc
            cur_y_te_enc = cur_enc.transform(cur_y_te) if not is_pca else y_te_enc

            for clf_name in classifiers:
                clf = classifier_params(clf_name, n_jobs=n_jobs, seed=RANDOM_SEED + int(fold))
                clf.fit(cur_X_tr, cur_y_tr_enc)
                pe = cur_enc.inverse_transform(np.asarray(clf.predict(cur_X_te), dtype=int))
                score = (clf.predict_proba(cur_X_te)[:, cur_pi] if hasattr(clf, "predict_proba")
                         else 1.0 / (1.0 + np.exp(-clf.decision_function(cur_X_te))) if hasattr(clf, "decision_function")
                         else np.asarray(clf.predict(cur_X_te) == cur_pi, dtype=float))

                # ── Save predictions.csv ──
                model_key = "baseline" if is_pca else method_name.replace("+", "_plus_")
                out_dir = OUTPUT_DIR / dataset_name / model_key / f"fold_{fold}" / clf_name
                out_dir.mkdir(parents=True, exist_ok=True)

                rows = [{"fold": int(fold), "sample_id": str(s), "y_true": str(yt), "y_pred": str(yp),
                         score_col: float(sc), "model": model_key, "paradigm": "real_baseline",
                         "variant_id": "baseline", "ratio": "", "alpha": "", "classifier": clf_name}
                        for s, yt, yp, sc in zip(cur_sid_te, cur_y_te, pe, score)]
                pd.DataFrame(rows).to_csv(out_dir / "predictions.csv", index=False)

                # ── Save packages.csv ──
                pkg_rows = []
                for sid in train_samples:
                    rs = subset_sample(real, sid)
                    pkg_rows.append({"sample_id": sid, "split": "train",
                                     "n_real": int(rs.n_obs), "n_synthetic": 0})
                for sid in test_samples:
                    rs = subset_sample(real, sid)
                    pkg_rows.append({"sample_id": sid, "split": "test",
                                     "n_real": int(rs.n_obs), "n_synthetic": 0})
                pd.DataFrame(pkg_rows).to_csv(out_dir / "packages.csv", index=False)

                # ── Save manifest.json ──
                manifest = {
                    "model": model_key,
                    "paradigm": "real_baseline",
                    "variant_id": "baseline",
                    "ratio": None,
                    "alpha": None,
                    "fold": int(fold),
                    "classifier": clf_name,
                    "n_train_samples": int(len(train_samples)),
                    "n_test_samples": int(len(test_samples)),
                    "quick_accuracy": round(float((pe == cur_y_te).mean()), 4),
                }
                (out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

                # ── Aggregate results ──
                RESULTS.setdefault(dataset_name, []).append({
                    "Dataset": dataset_name, "Fold": fold, "Classifier": clf_name, "Method": method_name,
                    "Accuracy": round(float((pe == cur_y_te).mean()), 4),
                    "BalAcc": round(balanced_accuracy_score(cur_y_te, pe), 4),
                    "MacroF1": round(f1_score(cur_y_te, pe, average="macro"), 4),
                    "MCC": round(matthews_corrcoef(cur_y_te, pe), 4),
                })
                print(f"    {method_name:20s} {clf_name}: Acc={float((pe == cur_y_te).mean()):.4f}")

df = pd.concat([pd.DataFrame(v) for v in RESULTS.values()], ignore_index=True)
print(f"\n{len(df)} aggregated rows in total, datasets: {df['Dataset'].unique()}")

In [ ]:
for ds in DATASETS:
    print(f"\n{'='*60}")
    print(f"  {ds} - cross-fold average results")
    print(f"{'='*60}")
    sub = df[df["Dataset"] == ds]
    display(sub.groupby(["Method", "Classifier"])[["Accuracy", "BalAcc", "MacroF1", "MCC"]].agg(["mean", "std"]).round(4))

In [ ]:
# Aggregate metrics summary
df.to_csv(OUTPUT_DIR / "per_run_results.csv", index=False)
for ds in DATASETS:
    sub = df[df["Dataset"] == ds]
    sub.groupby(["Method", "Classifier"])[["Accuracy", "BalAcc", "MacroF1", "MCC"]].agg(["mean", "std"]).round(4).to_csv(OUTPUT_DIR / f"summary_{ds}.csv")
print(f"\nResults saved to {OUTPUT_DIR.resolve()}/")
print()

for ds in DATASETS:
    print(f"  {ds}:")
    print(f"    {OUTPUT_DIR}/{ds}/baseline/<fold>/<classifier>/")
    for _, name, _ in EMBED_CONFIG:
        model_key = f"PCA+{name}".replace("+", "_plus_")
        print(f"    {OUTPUT_DIR}/{ds}/{model_key}/<fold>/<classifier>/")